# Module 08 — Token Embeddings

Modules 06-07's bigram model looked up rows in a counting table by
character *identity* — character 'k' and character 'q' are just different
indices, equally unrelated to every other pair of indices. That's what a
**one-hot vector** encodes: identity, and nothing else. There's no sense in
which 'a' and 'e' (both vowels, both behave similarly as neighbors) are any
more related than 'a' and 'x'.

A **token embedding** replaces "which index is this" with "a point in some
vector space" — a dense vector per token, initially random, that gets
*trained* so tokens which behave similarly end up near each other. This is
the very first layer of every neural language model, including nanoGPT and
the real pretrain this project is building toward.

## 1. One-hot vectors and their limitation

In [ ]:
import torch

vocab = list("abcdefghijklmnopqrstuvwxyz.")
stoi = {ch: i for i, ch in enumerate(vocab)}
vocab_size = len(vocab)

def one_hot(ch):
    v = torch.zeros(vocab_size)
    v[stoi[ch]] = 1.0
    return v

a_vec = one_hot("a")
x_vec = one_hot("x")
e_vec = one_hot("e")

# cosine similarity: 0 means "completely unrelated" in vector-space terms
cos = torch.nn.functional.cosine_similarity
print("similarity(a, x):", cos(a_vec.unsqueeze(0), x_vec.unsqueeze(0)).item())
print("similarity(a, e):", cos(a_vec.unsqueeze(0), e_vec.unsqueeze(0)).item())
print("Every pair of distinct one-hot vectors is equally (un)related - similarity is always 0.")

## 2. An embedding lookup is just a matrix multiply by a one-hot vector

Here's the key mechanical fact that makes embeddings click: if `E` is a
`(vocab_size, embed_dim)` matrix of learned vectors (one row per token),
then `one_hot(ch) @ E` picks out exactly row `stoi[ch]` of `E`. An
"embedding lookup" is mathematically a matrix multiply — it's just
implemented as plain indexing (`E[stoi[ch]]`) because that's astronomically
faster than actually multiplying by a mostly-zero vector.

In [ ]:
torch.manual_seed(42)
embed_dim = 2
E = torch.randn(vocab_size, embed_dim)

via_matmul = a_vec @ E
via_indexing = E[stoi["a"]]

assert torch.allclose(via_matmul, via_indexing)
print("one_hot(\'a\') @ E   =", via_matmul.tolist())
print("E[stoi[\'a\']]        =", via_indexing.tolist())
print("Identical, as expected - indexing IS the matmul, just done the fast way.")

`nn.Embedding` is exactly this: a `(vocab_size, embed_dim)` table of
learnable parameters, with fast indexing built in.

In [ ]:
import torch.nn as nn

embedding_layer = nn.Embedding(vocab_size, embed_dim)
embedding_layer.weight.data.copy_(E)  # use the same random init as above for a fair comparison

idx = torch.tensor(stoi["a"])
assert torch.allclose(embedding_layer(idx), E[stoi["a"]])
print("nn.Embedding(idx) matches E[idx] exactly - same lookup, framework-provided.")

## 3. Untrained embeddings are meaningless — training makes them useful

Right now `E` is random noise; 'a' and 'e' aren't any closer together than
'a' and 'x' — we just replaced one arbitrary representation (one-hot) with
another (random vectors). The vectors only become meaningful once we train
them on a task. Let's train a **2D** embedding (2D so we can plot it
directly) on the same next-character bigram-prediction task from Modules
06-07, using the Genshin character names, and see whether characters that
behave similarly as neighbors end up near each other.

In [ ]:
names = [
    "aether", "lumine", "amber", "kaeya", "lisa", "jean", "barbara", "diluc",
    "noelle", "bennett", "fischl", "sucrose", "chongyun", "klee", "xingqiu",
    "ningguang", "beidou", "xiangling", "xiao", "zhongli", "hutao", "yanfei",
    "rosaria", "albedo", "diona", "mona", "keqing", "qiqi", "venti",
    "tartaglia", "ganyu", "xinyan", "sayu", "kokomi", "kazuha", "ayaka",
    "yoimiya", "sara", "raiden", "aloy", "itto", "gorou", "yaemiko",
    "shinobu", "heizou", "yelan", "tighnari", "nahida", "nilou", "cyno",
    "candace", "layla", "wanderer", "faruzan", "dehya", "mika", "kaveh",
    "baizhu", "kirara", "lynette", "lyney", "freminet", "neuvillette",
    "wriothesley", "charlotte", "furina", "chevreuse", "navia", "chiori",
    "arlecchino", "clorinde", "sigewinne", "emilie", "kachina", "kinich",
    "mualani", "xilonen", "ororon", "chasca", "mavuika", "citlali", "varesa",
    "iansan", "escoffier", "ineffa",
]

# (input, target) pairs: every consecutive character bigram, wrapped in "."
inputs, targets = [], []
for name in names:
    wrapped = "." + name + "."
    for ch1, ch2 in zip(wrapped, wrapped[1:]):
        inputs.append(stoi[ch1])
        targets.append(stoi[ch2])
inputs = torch.tensor(inputs)
targets = torch.tensor(targets)
print(f"{len(inputs)} training bigrams")

In [ ]:
class TinyEmbeddingModel(nn.Module):
    def __init__(self, vocab_size, embed_dim):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.to_logits = nn.Linear(embed_dim, vocab_size)

    def forward(self, idx):
        return self.to_logits(self.embed(idx))


torch.manual_seed(42)
model = TinyEmbeddingModel(vocab_size, embed_dim=2)
optimizer = torch.optim.Adam(model.parameters(), lr=0.1)

for step in range(200):
    logits = model(inputs)
    loss = torch.nn.functional.cross_entropy(logits, targets)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if step % 40 == 0:
        print(f"step {step:3d}   loss {loss.item():.4f}")

print("final loss:", loss.item())

## 4. Plotting the trained embeddings

In [ ]:
import matplotlib.pyplot as plt

trained_E = model.embed.weight.detach()

plt.figure(figsize=(7, 7))
plt.scatter(trained_E[:, 0], trained_E[:, 1], s=0)
for ch, i in stoi.items():
    plt.text(trained_E[i, 0], trained_E[i, 1], ch, fontsize=12, ha="center", va="center")
plt.title("Trained 2D character embeddings")
plt.grid(True, alpha=0.3)
plt.show()

vowels = ["a", "e", "i", "o", "u"]
vowel_vecs = trained_E[[stoi[v] for v in vowels]]
avg_pairwise_dist_vowels = torch.pdist(vowel_vecs).mean().item()
all_vecs = trained_E[[stoi[c] for c in vocab if c != "."]]
avg_pairwise_dist_all = torch.pdist(all_vecs).mean().item()
print(f"Average distance between vowels:        {avg_pairwise_dist_vowels:.3f}")
print(f"Average distance between all characters: {avg_pairwise_dist_all:.3f}")

## Recap

- One-hot vectors encode identity only — every token is equally unrelated
  to every other.
- An embedding table is a `(vocab_size, embed_dim)` matrix of *learned*
  vectors; looking one up is mathematically `one_hot @ E`, implemented as
  fast indexing (`nn.Embedding`).
- Training reshapes the embedding space so that tokens behaving similarly
  in the task end up nearby — here, vowels end up closer to each other than
  the character population at large, purely as a side effect of learning to
  predict the next character.
- Real models use hundreds/thousands of embedding dimensions and much
  larger vocabularies (whole subword tokens, not single characters — see
  Module 19's BPE tokenizer), but the mechanism is identical to what's
  happening in this 2D toy example. Next up — Module 09: the neural n-gram
  model, which uses embeddings over a whole *window* of previous tokens
  (not just one) feeding into a real MLP.